# Kaggle CSVs and IMDB: Combining Datasets

Load a Kaggle disaster-tweets CSV from local disk, pull the IMDB movie-review dataset from HuggingFace, optimize a sentiment classifier on the public data, and then combine the public examples with a handful of domain-specific reviews.

**Required env vars:** `OPENAI_API_KEY` (loaded from `.env`).

**Optional local file:** `train.csv` from the Kaggle [disaster tweets](https://www.kaggle.com/competitions/nlp-getting-started/data) competition, placed next to this notebook. The Kaggle cell skips gracefully if the file is not present.

In [ ]:
%pip install -r ../requirements.txt -q
%pip install datasets -q

In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM('openai/gpt-5-mini'))

## Disaster tweets via Kaggle CSV

Kaggle requires accepting the competition rules and downloading the data manually. Place `train.csv` next to this notebook.

In [ ]:
import pandas as pd

csv_path = "train.csv"

if not os.path.exists(csv_path):
    print(f"Kaggle CSV not present at {csv_path}, skipping")
    kaggle_train_set = []
else:
    df = pd.read_csv(csv_path)

    # Convert to DSPy format
    kaggle_train_set = []
    for _, row in df.iterrows():
        kaggle_train_set.append(dspy.Example(
            text=row['text'],
            is_disaster=bool(row['target']),  # 0 or 1 -> False or True
        ).with_inputs('text'))

    print(kaggle_train_set[0].text)
    print(kaggle_train_set[0].is_disaster)

## IMDB reviews via HuggingFace

Use the IMDB movie-review dataset as a public sentiment corpus to prototype against.

In [ ]:
from datasets import load_dataset

# Load IMDB movie reviews
hf_dataset = load_dataset("imdb")
train_set = [
    dspy.Example(
        text=ex['text'],
        sentiment="positive" if ex['label'] == 1 else "negative",
    ).with_inputs('text')
    for ex in hf_dataset['train'].select(range(1000))
]

print(f"Loaded {len(train_set)} IMDB examples")
print(train_set[0].sentiment, '->', train_set[0].text[:120], '...')

## Optimize a sentiment classifier on the public data

A `BootstrapFewShot` pass picks high-quality demos from the full 1,000-example public corpus.

In [ ]:
class Sentiment(dspy.Signature):
    """Classify the sentiment of a review."""
    text: str = dspy.InputField()
    sentiment: str = dspy.OutputField(desc="positive or negative")

def exact_match(example, pred, trace=None):
    return example.sentiment.strip().lower() == pred.sentiment.strip().lower()

optimizer = dspy.BootstrapFewShot(metric=exact_match, max_bootstrapped_demos=8)
optimized = optimizer.compile(
    student=dspy.ChainOfThought(Sentiment),
    trainset=train_set,
)

## Add your own domain-specific examples

Replace the placeholder `custom_examples` below with reviews from your own product. The combined set lets the optimizer learn both general sentiment patterns and the vocabulary specific to your domain.

In [ ]:
# Placeholder for your own labelled product reviews
custom_examples = [
    dspy.Example(text="The new dashboard is buggy on mobile but the desktop view is great.",
                 sentiment="negative").with_inputs('text'),
    dspy.Example(text="Cut our onboarding time in half. Worth every penny.",
                 sentiment="positive").with_inputs('text'),
]

# Mix public + custom data
combined_train_set = train_set[:500] + custom_examples  # 500 public + custom

# Re-optimize on combined data
re_optimized = optimizer.compile(
    student=optimized,  # Start from the already-optimized program
    trainset=combined_train_set,
)

print("Re-optimized program is ready.")